# Stage 1: Cassava Disease Recognition - Gemma 3N Training

Entraînement de **Gemma 3N-E2B-it** (2B effective, 6B total) sur la reconnaissance des maladies du manioc.

## Pourquoi Gemma 3N ?
- ✅ **Plus rapide**: 4-6 heures vs 44h avec Qwen3-VL
- ✅ **Prouvé sur agriculture**: Modèle AgriVision existe
- ✅ **Meilleure efficacité**: 2B effective, moins de VRAM
- ✅ **Configuration optimisée**: LR 10x plus élevé, meilleur batch size

## Configuration Matérielle
- GPU: NVIDIA RTX 5090 32GB
- CUDA 12.9
- Precision: bfloat16

## Durée Estimée
- Entraînement: **4-6 heures** (3 époques)

---

## 1. Imports et Configuration

In [1]:
import os
import json
import re
import torch
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any
from datasets import Dataset
from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer
)
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers pour optimisations
from unsloth import FastVisionModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/tmp/ipykernel_9444/529373224.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


## 2. Configuration des Chemins et Paramètres

### Changements clés vs Qwen3-VL:
- **Modèle**: Gemma 3N-E2B-it (2B vs 8B)
- **Learning Rate**: 2e-4 (10x plus élevé !)
- **Batch size**: 4 (vs 1)
- **Grad accumulation**: 2 (vs 8)
- **Époques**: 3 (vs 5)

In [2]:
# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n_v2"  # v2 pour différencier
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle Gemma 3N
MODEL_NAME = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)  # Gemma 3N standard

# LoRA Configuration - OPTIMISÉE pour éviter overfitting
LORA_CONFIG = {
    "r": 8,                    # ↓↓↓ RÉDUIT de 32 à 8 (4x moins de params!)
    "lora_alpha": 16,          # Ajusté proportionnellement
    "lora_dropout": 0.15,      # ↑↑ AUGMENTÉ de 0.05 à 0.15 (3x plus de dropout)
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],  # Modules réduits
}

# Training Configuration - FORTE RÉGULARISATION
TRAINING_CONFIG = {
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 2,                   # ↓ RÉDUIT de 3 à 2 époques
    "learning_rate": 3e-5,                   # ↓↓↓ RÉDUIT de 2e-4 à 3e-5 (7x moins!)
    "warmup_steps": 200,                     # ↑ AUGMENTÉ de 100 à 200
    "weight_decay": 0.1,                     # ↑↑↑ AUGMENTÉ de 0.01 à 0.1 (10x plus!)
    "max_grad_norm": 0.5,                    # ↓ RÉDUIT de 1.0 à 0.5
    "logging_steps": 25,
    "save_steps": 500,
    "save_total_limit": 5,
    "eval_strategy": "steps",
    "eval_steps": 250,                       # ↑ Évaluer PLUS souvent (500 → 250)
    "fp16": not torch.cuda.is_bf16_supported(),
    "bf16": torch.cuda.is_bf16_supported(),
    "optim": "adamw_8bit",
    "lr_scheduler_type": "cosine",
    "seed": 42
}

print(f"\nBase Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"\n{'='*60}")
print(f"CONFIGURATION OPTIMISÉE ANTI-OVERFITTING")
print(f"{'='*60}")
print(f"\nModèle: {MODEL_NAME}")
print(f"Image size: {TARGET_IMAGE_SIZE}")
print(f"\n🔥 CHANGEMENTS CRITIQUES vs Config Précédente:")
print(f"\nLoRA:")
print(f"  - Rank (r): 8 (était 32) → 4x MOINS de paramètres")
print(f"  - Alpha: 16 (était 64)")
print(f"  - Dropout: 0.15 (était 0.05) → 3x PLUS de régularisation")
print(f"  - Target modules: 4 modules (était 7)")
print(f"\nTraining:")
print(f"  - Learning rate: 3e-5 (était 2e-4) → 7x PLUS LENT")
print(f"  - Weight decay: 0.1 (était 0.01) → 10x PLUS fort")
print(f"  - Époques: 2 (était 3) → Moins de temps pour overfitter")
print(f"  - Warmup: 200 steps (était 100)")
print(f"  - Eval freq: Tous les 250 steps (était 500)")
print(f"  - Max grad norm: 0.5 (était 1.0)")
print(f"\nEffective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"\n🎯 OBJECTIFS avec cette config:")
print(f"  - Train loss: 1.0-2.0 (PAS 0.02!)")
print(f"  - Val loss: 2.0-3.0")
print(f"  - Ratio train/val: 2:1 à 3:1")
print(f"  - Test accuracy: >60% (minimum) ou >75% (succès)")
print(f"{'='*60}")


Base Directory: /home/hounfodjidagba/learning/cassava/new_implementation
Data Directory: /home/hounfodjidagba/learning/cassava/new_implementation/data/stage1
Checkpoint Directory: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v2

CONFIGURATION OPTIMISÉE ANTI-OVERFITTING

Modèle: unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit
Image size: (512, 512)

🔥 CHANGEMENTS CRITIQUES vs Config Précédente:

LoRA:
  - Rank (r): 8 (était 32) → 4x MOINS de paramètres
  - Alpha: 16 (était 64)
  - Dropout: 0.15 (était 0.05) → 3x PLUS de régularisation
  - Target modules: 4 modules (était 7)

Training:
  - Learning rate: 3e-5 (était 2e-4) → 7x PLUS LENT
  - Weight decay: 0.1 (était 0.01) → 10x PLUS fort
  - Époques: 2 (était 3) → Moins de temps pour overfitter
  - Warmup: 200 steps (était 100)
  - Eval freq: Tous les 250 steps (était 500)
  - Max grad norm: 0.5 (était 1.0)

Effective batch size: 8

🎯 OBJECTIFS avec cette config:
  - Train loss: 1.0-2.0 (PAS 0.02!)

## 3. Chargement du Modèle avec Unsloth

Gemma 3N-E2B-it avec optimisations Unsloth:
- 2B effective parameters (6B total)
- 4-bit quantization
- Gradient checkpointing
- LoRA fine-tuning

In [3]:
print("\n" + "="*60)
print("LOADING GEMMA 3N WITH UNSLOTH")
print("="*60)

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

print(f"\nModel: {MODEL_NAME}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable %: {100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters()):.2f}%")
print("="*60)


LOADING GEMMA 3N WITH UNSLOTH
==((====))==  Unsloth 2026.1.1: Fast Gemma3N patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.15.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients

Model: unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit
Trainable parameters: 4,079,616
Total parameters: 4,511,595,968
Trainable %: 0.09%


## 4. Chargement des Datasets Cassava

Datasets préparés par `00_prepare_cassava_data.ipynb`:
- **Train**: 29,276 images équilibrées
- **Validation**: 3,612 images
- **Test**: 3,619 images

In [4]:
print("\n" + "="*60)
print("LOADING CASSAVA DATASETS")
print("="*60)

def load_json_dataset(file_path):
    """Charge un dataset JSON"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Charger les datasets cassava
cassava_train = load_json_dataset(DATA_DIR / "cassava_train_balanced.json")
cassava_val = load_json_dataset(DATA_DIR / "cassava_val.json")
cassava_test = load_json_dataset(DATA_DIR / "cassava_test.json")

print(f"Cassava Train (balanced): {len(cassava_train):,} samples")
print(f"Cassava Validation: {len(cassava_val):,} samples")
print(f"Cassava Test: {len(cassava_test):,} samples")
print(f"\nTotal: {len(cassava_train) + len(cassava_val) + len(cassava_test):,} samples")

# Charger les statistiques
stats_path = DATA_DIR / "dataset_stats.json"
if stats_path.exists():
    with open(stats_path, 'r') as f:
        stats = json.load(f)
    print("\nDistribution des classes (Train équilibré):")
    if 'train_balanced_distribution' in stats:
        for class_name, count in stats['train_balanced_distribution'].items():
            percentage = count / len(cassava_train) * 100
            print(f"  {class_name:40s}: {count:5d} ({percentage:5.2f}%)")


LOADING CASSAVA DATASETS
Cassava Train (balanced): 29,276 samples
Cassava Validation: 3,612 samples
Cassava Test: 3,619 samples

Total: 36,507 samples

Distribution des classes (Train équilibré):


## 5. Formatage des Données pour Gemma 3N

Conversion du format VQA vers le format attendu par Gemma 3N.
**Format identique à Qwen3-VL** - compatible!

In [5]:
def extract_image_path(text):
    """Extrait le chemin de l'image du tag <img>"""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def format_sample_for_gemma3n(sample):
    """Prépare un échantillon pour Gemma 3N (format compatible Qwen3-VL)"""
    conversations = sample['conversations']
    
    image_path = None
    formatted_conversations = []
    
    for turn in conversations:
        role = turn['from']
        content = turn['value']
        
        if '<img>' in content:
            img_path = extract_image_path(content)
            if img_path:
                image_path = img_path
            # Remplacer <img>path</img> par <image>
            content = re.sub(r'<img>.*?</img>', '<image>', content)
        
        formatted_conversations.append({
            "role": role,
            "content": content
        })
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id']
    }

print("\nFormatting datasets...")
train_formatted = [format_sample_for_gemma3n(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_for_gemma3n(s) for s in tqdm(cassava_val, desc="Val")]

# Créer datasets HuggingFace
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)

print(f"\nTrain dataset: {len(train_dataset):,} samples")
print(f"Eval dataset: {len(eval_dataset):,} samples")

# Afficher un exemple
print("\n--- Exemple formaté ---")
example = train_dataset[0]
print(f"ID: {example['id']}")
print(f"Image path: {example['image_path'][:80]}...")
print(f"Conversations:")
for i, conv in enumerate(example['conversations']):
    print(f"  [{i}] {conv['role']}: {conv['content'][:100]}...")


Formatting datasets...


Val: 100%|██████████| 3612/3612 [00:00<00:00, 597980.11it/s]


Train dataset: 29,276 samples
Eval dataset: 3,612 samples

--- Exemple formaté ---
ID: kaggle_3874993628.jpg
Image path: /home/hounfodjidagba/learning/cassava/data/raw/train_images/3874993628.jpg...
Conversations:
  [0] user: Picture 1: <image>
What cassava disease is shown in this image?...
  [1] assistant: Disease: Cassava Mosaic Disease (CMD)

Cassava Mosaic Disease (CMD) is one of the most devastating c...


## 6. Data Collator pour Gemma 3N

### 🔴 CORRECTIONS CRITIQUES vs Qwen3-VL:
1. **Resize images à (512, 512)** - OBLIGATOIRE pour Gemma 3N
2. **Truncation activée** avec max_length=1024 - évite OOM
3. **Gestion erreurs images** - fallback si image corrompue

In [6]:
@dataclass
class Gemma3NDataCollator:
    """Data collator pour Gemma 3N - VERSION CORRIGÉE - Fix labels masking"""
    processor: Any
    base_dir: str = str(BASE_DIR.parent)

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path:
                try:
                    if os.path.exists(image_path):
                        image = Image.open(image_path).convert('RGB')
                        # ✅ CRITIQUE: Resize à (512, 512) pour Gemma 3N
                        image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                    else:
                        print(f"Warning: Image not found: {image_path[:80]}...")
                        image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
                except Exception as e:
                    print(f"Error loading image {image_path[:80]}...: {e}")
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)

            # Construire messages
            conversations = feature['conversations']
            messages = []
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        # ✅ CORRECTION 1 : Créer textes COMPLETS et USER-ONLY séparément
        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            # Texte complet avec réponse (pour input_ids)
            full_text = self.processor.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            # Texte USER ONLY (pour calculer la longueur à masquer)
            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages,
                tokenize=False,
                add_generation_prompt=True  # Ajoute le token <|im_start|>assistant
            )
            batch_texts_user.append(user_text)

        # ✅ CORRECTION 2 : Processeur appelé UNE SEULE FOIS avec images
        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        # ✅ CORRECTION 3 : Masquage CORRECT des labels
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            # Tokenizer la partie USER uniquement (avec images!)
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],  # IMPORTANT : inclure l'image
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=1024
            )

            # Longueur EXACTE de la partie user (avec image tokens)
            user_length = user_encoding["input_ids"].shape[1]

            # ✅ Masquer UNIQUEMENT les tokens user (question + image)
            # Garder les tokens assistant (réponse) pour la loss
            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                # Si user_length >= total, c'est un problème de troncature
                # Dans ce cas, masquer presque tout mais garder au moins quelques tokens
                labels[i, :-10] = -100  # Garder les 10 derniers tokens pour avoir un signal

            # ✅ VÉRIFICATION : S'assurer qu'il reste des tokens actifs
            active_tokens = (labels[i] != -100).sum().item()
            if active_tokens < 5:
                print(f"⚠️  WARNING Sample {i}: Seulement {active_tokens} tokens actifs pour la loss!")

        batch["labels"] = labels

        return batch

data_collator = Gemma3NDataCollator(processor=processor, base_dir=str(BASE_DIR.parent))

print("✅ Data collator CORRIGÉ créé")
print(f"Base directory for images: {BASE_DIR.parent}")
print(f"Target image size: {TARGET_IMAGE_SIZE}")
print(f"Max sequence length: 1024 tokens")
print(f"Truncation: ENABLED")
print(f"")
print(f"🔥 CORRECTIONS CRITIQUES APPLIQUÉES:")
print(f"  1. Textes USER-ONLY et COMPLETS séparés")
print(f"  2. Images incluses dans calcul user_length")
print(f"  3. Fallback: garder 10 tokens si troncature")
print(f"  4. Vérification: warning si < 5 tokens actifs")

✅ Data collator CORRIGÉ créé
Base directory for images: /home/hounfodjidagba/learning/cassava
Target image size: (512, 512)
Max sequence length: 1024 tokens
Truncation: ENABLED

🔥 CORRECTIONS CRITIQUES APPLIQUÉES:
  1. Textes USER-ONLY et COMPLETS séparés
  2. Images incluses dans calcul user_length
  3. Fallback: garder 10 tokens si troncature
  4. Vérification: warning si < 5 tokens actifs


In [7]:
# 🔍 DEBUG: Vérifier le masquage des labels

print("="*60)
print("VÉRIFICATION DU DATA COLLATOR")
print("="*60)

# Créer un batch de test
test_features = [train_dataset[0], train_dataset[1], train_dataset[2], train_dataset[3]]
test_batch = data_collator(test_features)

print(f"\nBatch créé:")
print(f"  - input_ids shape: {test_batch['input_ids'].shape}")
print(f"  - labels shape: {test_batch['labels'].shape}")

# Analyser chaque sample
for i in range(len(test_features)):
    print(f"\n{'='*60}")
    print(f"SAMPLE {i}: {test_features[i]['id']}")
    print(f"{'='*60}")

    input_ids = test_batch["input_ids"][i]
    labels = test_batch["labels"][i]

    # Compter tokens
    total_tokens = (input_ids != processor.tokenizer.pad_token_id).sum().item()
    masked_tokens = (labels == -100).sum().item()
    active_tokens = ((labels != -100) & (labels != processor.tokenizer.pad_token_id)).sum().item()

    print(f"\nStatistiques tokens:")
    print(f"  Total tokens (non-padding): {total_tokens}")
    print(f"  Tokens masqués (-100): {masked_tokens} ({masked_tokens/total_tokens*100:.1f}%)")
    print(f"  Tokens actifs (loss): {active_tokens} ({active_tokens/total_tokens*100:.1f}%)")

    # Décoder les tokens actifs
    active_indices = ((labels != -100) & (labels != processor.tokenizer.pad_token_id)).nonzero(as_tuple=True)[0]
    if len(active_indices) > 0:
        active_token_ids = input_ids[active_indices]
        decoded_active = processor.tokenizer.decode(active_token_ids, skip_special_tokens=False)
        print(f"\n✅ Tokens actifs (devrait être la réponse de l'assistant):")
        print(f"   {decoded_active[:300]}")
    else:
        print(f"\n❌ ERREUR: AUCUN TOKEN ACTIF! Tous les tokens sont masqués!")

    # Décoder les tokens masqués
    masked_indices = ((labels == -100) & (input_ids != processor.tokenizer.pad_token_id)).nonzero(as_tuple=True)[0]
    if len(masked_indices) > 0:
        masked_token_ids = input_ids[masked_indices]
        decoded_masked = processor.tokenizer.decode(masked_token_ids, skip_special_tokens=False)
        print(f"\n✅ Tokens masqués (devrait être la question + image):")
        print(f"   {decoded_masked[:300]}")

print(f"\n{'='*60}")
print(f"RÉSUMÉ")
print(f"{'='*60}")

# Vérifications globales
all_active = [(test_batch["labels"][i] != -100).sum().item() for i in range(len(test_features))]
min_active = min(all_active)
max_active = max(all_active)
avg_active = sum(all_active) / len(all_active)

print(f"\nTokens actifs par sample:")
print(f"  - Min: {min_active}")
print(f"  - Max: {max_active}")
print(f"  - Moyenne: {avg_active:.1f}")

if min_active < 5:
    print(f"\n❌ ALERTE: Au moins 1 sample a < 5 tokens actifs!")
    print(f"   → Le data collator a ENCORE un problème")
elif avg_active < 30:
    print(f"\n⚠️  WARNING: Moyenne < 30 tokens actifs par sample")
    print(f"   → Vérifier si c'est normal pour vos données")
else:
    print(f"\n✅ OK: Tous les samples ont un nombre raisonnable de tokens actifs")

print(f"\n{'='*60}")

VÉRIFICATION DU DATA COLLATOR

Batch créé:
  - input_ids shape: torch.Size([4, 332])
  - labels shape: torch.Size([4, 332])

SAMPLE 0: kaggle_3874993628.jpg

Statistiques tokens:
  Total tokens (non-padding): 331
  Tokens masqués (-100): 284 (85.8%)
  Tokens actifs (loss): 48 (14.5%)

✅ Tokens actifs (devrait être la réponse de l'assistant):
   
Disease: Cassava Mosaic Disease (CMD)

Cassava Mosaic Disease (CMD) is one of the most devastating cassava diseases, causing severe mosaic patterns, leaf distortion, and stunted growth. It is transmitted by whiteflies.<end_of_turn>


✅ Tokens masqués (devrait être la question + image):
   <bos><bos><start_of_turn>user


<start_of_image><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token>

SAMPLE 1: cbsd-1_1617103815345

Statistiques tokens:
  Total

## 7. Configuration du Trainer

In [8]:
print("\n" + "="*60)
print("CONFIGURING TRAINER")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["save_steps"],
    fp16=TRAINING_CONFIG["fp16"],
    bf16=TRAINING_CONFIG["bf16"],
    optim=TRAINING_CONFIG["optim"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    seed=TRAINING_CONFIG["seed"],
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

effective_batch_size = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch_size * TRAINING_CONFIG['num_train_epochs']

print(f"\nTraining Configuration:")
print(f"  - Device batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"  - Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  - Effective batch size: {effective_batch_size}")
print(f"  - Total training steps: {total_steps:,}")
print(f"  - Warmup steps: {TRAINING_CONFIG['warmup_steps']}")
print(f"  - Logging every: {TRAINING_CONFIG['logging_steps']} steps")
print(f"  - Evaluation every: {TRAINING_CONFIG['save_steps']} steps")
print(f"  - Learning rate: {TRAINING_CONFIG['learning_rate']} (10x Qwen3-VL!)")
print(f"  - Precision: {'bfloat16' if TRAINING_CONFIG['bf16'] else 'float16'}")
print("="*60)


CONFIGURING TRAINER

Training Configuration:
  - Device batch size: 4
  - Gradient accumulation: 2
  - Effective batch size: 8
  - Total training steps: 7,318
  - Warmup steps: 200
  - Logging every: 25 steps
  - Evaluation every: 500 steps
  - Learning rate: 3e-05 (10x Qwen3-VL!)
  - Precision: bfloat16


/tmp/ipykernel_9444/2911375720.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(


## 8. Entraînement

### 🎯 Objectifs
- **Step 500**: Loss < 3.0
- **Epoch 1**: Loss < 1.5
- **Final**: Loss < 1.0, Accuracy >85%

### ⏱️ Temps Estimé
- **4-6 heures** sur RTX 5090
- Vitesse: ~2-3 it/sec

### ⚠️ Monitoring
**ARRÊTER SI**: Loss > 5.0 après 1000 steps

In [9]:
print("\n" + "="*60)
print("STARTING TRAINING - STAGE 1: GEMMA 3N CASSAVA")
print("="*60)
print(f"Estimated time: 4-6 hours on RTX 5090")
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(eval_dataset):,}")
print(f"Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"")
print(f"🎯 Success Criteria:")
print(f"  - Step 500: loss < 3.0")
print(f"  - Epoch 1: loss < 1.5")
print(f"  - Final: loss < 1.0")
print(f"")
print(f"⚠️  STOP if loss > 5.0 after 1000 steps")
print("="*60)

# Lancer l'entraînement
trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 29,276 | Num Epochs = 2 | Total steps = 7,320
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 4,079,616 of 5,443,517,888 (0.07% trained)



STARTING TRAINING - STAGE 1: GEMMA 3N CASSAVA
Estimated time: 4-6 hours on RTX 5090
Training samples: 29,276
Validation samples: 3,612
Epochs: 2

🎯 Success Criteria:
  - Step 500: loss < 3.0
  - Epoch 1: loss < 1.5
  - Final: loss < 1.0

⚠️  STOP if loss > 5.0 after 1000 steps
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,0.027500,2.990531
1000,0.021600,2.857908
1500,0.024800,2.855510
2000,0.023600,2.999447
2500,0.017400,2.958258
3000,0.017800,2.787441
3500,0.014400,2.784642
4000,0.018500,2.847145
4500,0.018700,2.820520
5000,0.020300,2.769892


Unsloth: Not an error, but Gemma3nForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_005/uuid_050e595c-1c60-4ceb-8ca5-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_003/uuid_5a8bfda6-3611-4da9-86f9-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_0adc005b-d4ee-4700-b7ef-...: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_0adc005b-d4ee-4700-b7ef-85c8eff62d54.jpg'
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_005/uuid_050e595c-1c60-4ceb-8ca5-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_003/uuid_5a8bfda6-3611-4da9-86f9-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_2e71b426-3fd4-4320-a615-...: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/c

In [11]:
# 📌 TEST FINAL - FIN EPOCH 2 (avec formatage du dataset)
print("="*60)
print("TEST FINAL - FIN EPOCH 2")
print("="*60)

# ✅ ÉTAPE 1 : Formater le dataset de test (si pas encore fait)
if 'test_formatted' not in locals():
    print("\nFormatage du dataset de test...")
    test_formatted = [format_sample_for_gemma3n(s) for s in tqdm(cassava_test, desc="Test")]
    print(f"Test dataset formaté: {len(test_formatted):,} samples\n")

# ✅ ÉTAPE 2 : Préparer le modèle pour l'inférence
FastVisionModel.for_inference(model)

# ✅ ÉTAPE 3 : Sélectionner 100 samples aléatoires
import random
from collections import Counter, defaultdict
random.seed(42)
test_subset = random.sample(test_formatted, 100)

print(f"Test sur {len(test_subset)} samples...")

# ✅ ÉTAPE 4 : Tester
predictions = []
ground_truths = []
results = []

for idx, sample in enumerate(tqdm(test_subset, desc="Testing")):
    # Préparer l'image
    image = Image.open(sample["image_path"]).convert("RGB")
    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
    
    # Construire le message
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Analyze this cassava leaf image and determine which disease is present. Provide your analysis in this format:\n\nDisease: [disease name]\n\n[Brief explanation]"}
            ]
        }
    ]
    
    # Préparer l'input
    input_text = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[input_text], images=[[image]], return_tensors="pt", padding=True).to(model.device)
    
    # Générer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    
    # Décoder
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extraire la maladie prédite
    predicted_disease = "UNKNOWN"
    if "Disease:" in response:
        disease_line = response.split("Disease:")[1].split("\n")[0].strip()
        predicted_disease = disease_line
    
    # Vérité terrain
    gt_disease = None
    for conv in sample["conversations"]:
        if conv["role"] == "assistant":
            gt_text = conv["content"]
            if "Disease:" in gt_text:
                gt_disease = gt_text.split("Disease:")[1].split("\n")[0].strip()
    
    predictions.append(predicted_disease)
    ground_truths.append(gt_disease)
    
    # Vérifier correspondance
    is_correct = gt_disease and predicted_disease.lower() in gt_disease.lower()
    results.append({
        "id": sample["id"],
        "predicted": predicted_disease,
        "ground_truth": gt_disease,
        "correct": is_correct
    })

# ✅ ÉTAPE 5 : Calculer les résultats
correct = sum(1 for r in results if r["correct"])
accuracy = correct / len(test_subset) * 100

print(f"\n{'='*60}")
print(f"RÉSULTATS FINAUX - EPOCH 2")
print(f"{'='*60}")
print(f"Accuracy: {accuracy:.1f}% ({correct}/{len(test_subset)})")

# Distribution des prédictions
pred_counts = Counter(predictions)
print(f"\nDistribution des prédictions:")
for disease, count in pred_counts.most_common():
    print(f"  {disease}: {count}")

# Accuracy par classe
class_stats = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results:
    if r["ground_truth"]:
        # Extraire la classe principale (CMD, CBSD, etc.)
        gt_class = r["ground_truth"].split("(")[0].strip() if "(" in r["ground_truth"] else r["ground_truth"]
        class_stats[gt_class]["total"] += 1
        if r["correct"]:
            class_stats[gt_class]["correct"] += 1

print(f"\n{'='*60}")
print(f"ACCURACY PAR CLASSE")
print(f"{'='*60}")
for class_name in sorted(class_stats.keys()):
    stats = class_stats[class_name]
    acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
    print(f"  {class_name:40s}: {stats['correct']:2d}/{stats['total']:2d} = {acc:5.1f}%")

# Exemples d'erreurs
errors = [r for r in results if not r["correct"]]
if errors:
    print(f"\n{'='*60}")
    print(f"EXEMPLES D'ERREURS (5 premiers)")
    print(f"{'='*60}")
    for i, err in enumerate(errors[:5]):
        print(f"\n  [{i+1}] ID: {err['id']}")
        print(f"      Ground Truth: {err['ground_truth']}")
        print(f"      Prédiction  : {err['predicted']}")

# Décision finale
print(f"\n{'='*60}")
print(f"DÉCISION FINALE")
print(f"{'='*60}")

if accuracy >= 75:
    print(f"✅ SUCCÈS COMPLET ! Accuracy {accuracy:.1f}% >= 75%")
    print(f"   → Stage 1 terminé avec succès")
    print(f"   → Prêt pour Stage 2 (multi-tour) ou Stage 4 (français)")
elif accuracy >= 60:
    print(f"✅ SUCCÈS ! Accuracy {accuracy:.1f}% >= 60%")
    print(f"   → Objectif minimum atteint")
    print(f"   → Possibilité d'optimiser davantage si besoin")
elif accuracy >= 40:
    print(f"⚠️  Accuracy {accuracy:.1f}% entre 40-60%")
    print(f"   → Acceptable mais en dessous de l'objectif")
    print(f"   → Envisager ajustements (r=4, LR=2e-5)")
else:
    print(f"❌ Accuracy {accuracy:.1f}% < 40%")
    print(f"   → Besoin d'ajustements supplémentaires")

print(f"\n{'='*60}")


TEST FINAL - FIN EPOCH 2

Formatage du dataset de test...


Test: 100%|██████████| 3619/3619 [00:00<00:00, 638424.72it/s]


Test dataset formaté: 3,619 samples

Test sur 100 samples...


Testing: 100%|██████████| 100/100 [08:07<00:00,  4.87s/it]


RÉSULTATS FINAUX - EPOCH 2
Accuracy: 36.0% (36/100)

Distribution des prédictions:
  Cassava Mosaic Disease (CMD): 27
  Mosaic: 13
  Mosaic Disease: 8
  Leaf Spot: 7
  Mosaic Virus (PVY): 6
  Cassava Leaf Chlorosis: 3
  Leaf Chlorosis: 3
  Mosaic Virus (CPMV): 3
  Cassava Leaf Blight (Caused by *Colletotrichum orangei*): 3
  Cassava Yellowing: 3
  Cassava Leaf Mosaic Disease (CLMD): 2
  Anthracnose: 2
  Xanthomonas tropica (Cassava Bacterial Blight): 2
  None Detected: 2
  Leaf Tip Burn: 2
  Leaf Tip Blister: 1
  Leaf Tip Burn/Scorch: 1
  Cassava Streak: 1
  Cassava Leaf Blight (Caused by *Colletotrichum spp.*): 1
  Cassava Leaf Blight (Caused by *Cercospora cassavae*): 1
  Tomato Mosaic Disease (TMD): 1
  Bacterial Leaf Spot (Bacterial Leaf Spot): 1
  Cassava Rot (caused by Phytophthora infestans): 1
  Leaf Mosaic Disease (LM): 1
  Leaf Mosaic Disease (LMD): 1
  Cassava Leaf Blight (CLB): 1
  Chlorosis: 1
  Cassava Leaf Blight (caused by *Colletotrichum spp.*): 1
  Cassava Leaf Bligh

In [12]:
# 📌 FORMATER LE DATASET DE TEST (si pas encore fait)
print("="*60)
print("FORMATAGE DU DATASET DE TEST")
print("="*60)

# Formater le dataset de test avec la fonction déjà définie
test_formatted = [format_sample_for_gemma3n(s) for s in tqdm(cassava_test, desc="Test")]
print(f"Test dataset formaté: {len(test_formatted):,} samples")

# ⏸️ TEST INTERMÉDIAIRE À STEP 2000
print("\n" + "="*60)
print("TEST INTERMÉDIAIRE - STEP 2000")
print("="*60)

# Préparer le modèle pour l'inférence
FastVisionModel.for_inference(model)

# Sélectionner 20 samples aléatoires
import random
from collections import Counter
random.seed(42)
test_subset = random.sample(test_formatted, 20)

# Tester
predictions = []
ground_truths = []

for idx, sample in enumerate(test_subset):
    print(f"\n{'='*60}")
    print(f"TEST {idx+1}/20 - ID: {sample['id']}")
    print(f"{'='*60}")
    
    # Préparer l'image
    image = Image.open(sample["image_path"]).convert("RGB")
    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
    
    # Construire le message
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Analyze this cassava leaf image and determine which disease is present. Provide your analysis in this format:\n\nDisease: [disease name]\n\n[Brief explanation]"}
            ]
        }
    ]
    
    # Préparer l'input
    input_text = processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True
    ).to(model.device)
    
    # Générer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    
    # Décoder
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extraire la maladie prédite
    predicted_disease = "UNKNOWN"
    if "Disease:" in response:
        disease_line = response.split("Disease:")[1].split("\n")[0].strip()
        predicted_disease = disease_line
    
    # Vérité terrain
    gt_disease = None
    for conv in sample["conversations"]:
        if conv["role"] == "assistant":
            gt_text = conv["content"]
            if "Disease:" in gt_text:
                gt_disease = gt_text.split("Disease:")[1].split("\n")[0].strip()
    
    predictions.append(predicted_disease)
    ground_truths.append(gt_disease)
    
    print(f"Ground Truth: {gt_disease}")
    print(f"Prédiction  : {predicted_disease}")
    print(f"Réponse     : {response[:200]}")
    
    # Vérifier si c'est du charabia
    if "What cassava" in response or response.count("This cassava") > 2:
        print("❌ ALERTE: CHARABIA DÉTECTÉ!")

# Calculer l'accuracy
correct = sum(1 for p, g in zip(predictions, ground_truths) if g and p.lower() in g.lower())
accuracy = correct / len(test_subset) * 100

print(f"\n{'='*60}")
print(f"RÉSULTATS STEP 2000")
print(f"{'='*60}")
print(f"Accuracy: {accuracy:.1f}% ({correct}/{len(test_subset)})")
print(f"\nDistribution des prédictions:")
pred_counts = Counter(predictions)
for disease, count in pred_counts.most_common():
    print(f"  {disease}: {count}")

print(f"\n{'='*60}")
print(f"DÉCISION")
print(f"{'='*60}")

if accuracy >= 40 and len(pred_counts) > 2:
    print("✅ CONTINUER jusqu'à epoch 2")
    print("   - Accuracy > 40% ✅")
    print("   - Diversité des prédictions ✅")
elif accuracy < 40:
    print("❌ ARRÊTER L'ENTRAÎNEMENT")
    print(f"   - Accuracy trop basse: {accuracy:.1f}% < 40%")
    print("   - Action: Réduire encore (r=4, LR=1e-5)")
elif len(pred_counts) == 1:
    print("❌ ARRÊTER L'ENTRAÎNEMENT")
    print("   - Modèle prédit toujours la même classe")
    print("   - Action: Vérifier le dataset ou réduire capacité")


FORMATAGE DU DATASET DE TEST


Test: 100%|██████████| 3619/3619 [00:00<00:00, 580621.44it/s]

Test dataset formaté: 3,619 samples

TEST INTERMÉDIAIRE - STEP 2000

TEST 1/20 - ID: cbsd-2_1617175252584


Ground Truth: Cassava Brown Streak Disease (CBSD)
Prédiction  : Cassava Leaf Blight
Réponse     : Disease: Cassava Leaf Blight

Cassava Leaf Blight is a fungal disease that causes brown spots and lesions on leaves, stems, and roots. It can significantly reduce crop yields if left untreated. 

**Sy

TEST 2/20 - ID: kaggle_2552409953.jpg
Ground Truth: Cassava Mosaic Disease (CMD)
Prédiction  : Cassava Mosaic Disease (CMD)
Réponse     : Disease: Cassava Mosaic Disease (CMD)

Cassava Mosaic Disease (CMD) is a viral disease that affects cassava plants. It causes symptoms like yellowing, mottling, and leaf distortion. It is spread throu

TEST 3/20 - ID: cassava_cbb_train-cbb-442
Ground Truth: Cassava Bacterial Blight (CBB)
Prédiction  : Anthracnose
Réponse     : Disease: Anthracnose

Anthracnose is a fungal disease that causes dark, sunken lesions on leaves, stems, and fruits. It can lead to significant yield losses if not managed. 

**Symptoms:** Dark brown 

TEST 4/20 - ID: kaggle_22069946

In [11]:
# 🧪 TEST RAPIDE: Accuracy sur 100 samples du test set
import random
import re

# Sauvegarder checkpoint actuel
print("Sauvegarde du checkpoint step 6000...")
trainer.save_model(str(CHECKPOINT_DIR / "checkpoint-6000"))
print(f"✅ Sauvegardé dans: {CHECKPOINT_DIR / 'checkpoint-6000'}")

# Fonction pour extraire image path
def extract_image_path(text):
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

# Charger test set
print("\nChargement du test set...")
test_samples = load_json_dataset(DATA_DIR / "cassava_test.json")
random.seed(42)
random.shuffle(test_samples)
test_subset = test_samples[:100]
print(f"Test sur {len(test_subset)} samples")

# Préparer modèle pour inférence
FastVisionModel.for_inference(model)

# Évaluer
correct = 0
predictions_detail = []

for sample in tqdm(test_subset, desc="Testing"):
    # Extraire infos
    image_path = extract_image_path(sample['conversations'][0]['value'])
    true_label = sample['metadata']['label']  # 0-4
    true_class = sample['metadata']['class_name']
    
    # Charger image
    image = Image.open(image_path).convert('RGB').resize((512, 512))
    
    # Question
    question = "What cassava disease is shown in this image?"
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]
    
    # Générer prédiction
    text = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[[image]], return_tensors="pt", padding=True).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=False)
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire disease name (simple heuristic)
    response_upper = response.upper()
    if "CBB" in response_upper or "BACTERIAL BLIGHT" in response_upper:
        pred_label = 0
        pred_class = "CBB"
    elif "CBSD" in response_upper or "BROWN STREAK" in response_upper:
        pred_label = 1
        pred_class = "CBSD"
    elif "CGM" in response_upper or "GREEN MOTTLE" in response_upper:
        pred_label = 2
        pred_class = "CGM"
    elif "CMD" in response_upper or "MOSAIC" in response_upper:
        pred_label = 3
        pred_class = "CMD"
    elif "HEALTHY" in response_upper or "NO DISEASE" in response_upper:
        pred_label = 4
        pred_class = "Healthy"
    else:
        # Fallback: prédire la classe la plus fréquente (CMD)
        pred_label = 3
        pred_class = "CMD (fallback)"
    
    is_correct = (pred_label == true_label)
    if is_correct:
        correct += 1
    
    predictions_detail.append({
        "true_label": true_label,
        "pred_label": pred_label,
        "true_class": true_class,
        "pred_class": pred_class,
        "correct": is_correct,
        "response": response[:200]  # Premier 200 chars
    })

# Résultats
accuracy = correct / len(test_subset)
print("\n" + "="*60)
print("🎯 RÉSULTATS TEST - Step 6000")
print("="*60)
print(f"Accuracy: {accuracy:.1%} ({correct}/{len(test_subset)})")

# Accuracy par classe
print("\n📊 Accuracy par classe:")
from collections import defaultdict
class_stats = defaultdict(lambda: {"correct": 0, "total": 0})

for pred in predictions_detail:
    true_class = pred["true_class"].split("(")[0].strip()  # Enlever (CBB) etc
    class_stats[true_class]["total"] += 1
    if pred["correct"]:
        class_stats[true_class]["correct"] += 1

for class_name in sorted(class_stats.keys()):
    stats = class_stats[class_name]
    acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
    print(f"  {class_name:40s}: {stats['correct']:2d}/{stats['total']:2d} = {acc:5.1f}%")

# Exemples d'erreurs
print("\n❌ Exemples d'erreurs (5 premiers):")
errors = [p for p in predictions_detail if not p["correct"]]
for i, err in enumerate(errors[:5]):
    print(f"\n  [{i+1}] True: {err['true_class']:20s} | Pred: {err['pred_class']:20s}")
    print(f"      Response: {err['response'][:100]}...")

# DÉCISION
print("\n" + "="*60)
print("🎯 DÉCISION RECOMMANDÉE")
print("="*60)
if accuracy >= 0.75:
    print(f"✅ Accuracy {accuracy:.1%} >= 75%")
    print("\n👉 RECOMMANDATION: CONTINUER l'entraînement (Option B)")
    print("   - Le modèle généralise correctement malgré val loss élevée")
    print("   - Laisser terminer epoch 2 (~1h30 restant)")
    print("   - Sauvegarder best model (probablement step 3500-4000)")
else:
    print(f"❌ Accuracy {accuracy:.1%} < 75%")
    print("\n👉 RECOMMANDATION: RELANCER avec config optimisée (Option A)")
    print("   - Vraie mauvaise généralisation détectée")
    print("   - Modifier config: LR=5e-5, r=16, weight_decay=0.05, epochs=2")
    print("   - Relancer entraînement (~3-4h)")

print("="*60)

Sauvegarde du checkpoint step 6000...


✅ Sauvegardé dans: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v2/checkpoint-6000

Chargement du test set...
Test sur 100 samples


Testing:   6%|▌         | 6/100 [01:10<18:22, 11.73s/it]


KeyboardInterrupt: 

In [9]:
# Sauvegarder checkpoint actuel
trainer.save_model(str(CHECKPOINT_DIR / "checkpoint-6000"))

# Évaluation rapide sur 100 samples
import random
from tqdm import tqdm

# Charger test set
test_samples = load_json_dataset(DATA_DIR / "cassava_test.json")
random.shuffle(test_samples)
test_subset = test_samples[:100]

# Préparer pour inférence
FastVisionModel.for_inference(model)

# Évaluer
correct = 0
for sample in tqdm(test_subset, desc="Testing"):
    image_path = sample['image_path']
    true_label = sample['label']  # 0-4
    
    # Charger image
    image = Image.open(image_path).convert('RGB').resize((512, 512))
    
    # Question
    question = "What cassava disease is shown in this image?"
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]
    
    # Générer prédiction
    text = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[[image]], return_tensors="pt", padding=True).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire disease name (simple heuristic)
    if "CBB" in response or "Bacterial Blight" in response:
        pred_label = 0
    elif "CBSD" in response or "Brown Streak" in response:
        pred_label = 1
    elif "CGM" in response or "Green Mottle" in response:
        pred_label = 2
    elif "CMD" in response or "Mosaic" in response:
        pred_label = 3
    else:
        pred_label = 4  # Healthy
    
    if pred_label == true_label:
        correct += 1

accuracy = correct / len(test_subset)
print(f"\n🎯 Test Accuracy (100 samples): {accuracy:.1%}")
print(f"Correct: {correct}/100")


Testing:   0%|          | 0/100 [00:00<?, ?it/s]


KeyError: 'image_path'

## 9. Sauvegarde du Modèle Final

In [ ]:
print("\n" + "="*60)
print("SAVING FINAL MODEL")
print("="*60)

# Sauvegarder le modèle complet
final_model_path = CHECKPOINT_DIR / "final_model"
trainer.save_model(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

# Sauvegarder les adapters LoRA séparément
lora_adapters_path = CHECKPOINT_DIR / "lora_adapters"
model.save_pretrained(str(lora_adapters_path))

# Sauvegarder les métriques d'entraînement
metrics = {
    "stage": "stage1_cassava_gemma3n",
    "model": MODEL_NAME,
    "train_samples": len(train_dataset),
    "eval_samples": len(eval_dataset),
    "epochs": TRAINING_CONFIG["num_train_epochs"],
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "final_eval_loss": trainer.state.log_history[-1].get('eval_loss', None) if trainer.state.log_history else None,
}

with open(CHECKPOINT_DIR / "training_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Modèle complet sauvegardé: {final_model_path}")
print(f"✅ Adapters LoRA sauvegardés: {lora_adapters_path}")
print(f"✅ Métriques sauvegardées: {CHECKPOINT_DIR / 'training_metrics.json'}")
print("\n" + "="*60)
print("STAGE 1 GEMMA 3N COMPLETED SUCCESSFULLY")
print("="*60)

## 10. Résumé et Prochaines Étapes

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY - GEMMA 3N")
print("="*60)

# Afficher l'historique d'entraînement
if trainer.state.log_history:
    print("\nDernières métriques:")
    for entry in trainer.state.log_history[-5:]:
        step = entry.get('step', 'N/A')
        loss = entry.get('loss', 'N/A')
        eval_loss = entry.get('eval_loss', 'N/A')
        print(f"  Step {step}: train_loss={loss}, eval_loss={eval_loss}")

print("\n" + "="*60)
print("PROCHAINES ÉTAPES")
print("="*60)
print("""
1. Évaluer le modèle sur le test set (cassava_test.json)
   - Calculer accuracy par classe
   - Matrice de confusion
   - Vérifier >85% accuracy globale

2. Si résultats satisfaisants:
   - Passer au Stage 2 (optionnel): Conversations multi-tour
   - Passer au Stage 4: Support français
   - Déploiement

3. Si résultats insuffisants:
   - Analyser les erreurs par classe
   - Essayer Gemma 3N-E4B-it (4B) pour meilleure qualité
   - Ajuster hyperparamètres

Fichiers générés:
  - {}
  - {}
  - {}
""".format(final_model_path, lora_adapters_path, CHECKPOINT_DIR / 'training_metrics.json'))

print("="*60)
print("\n🎉 Avantages Gemma 3N vs Qwen3-VL:")
print("  ✅ 4-6 heures vs 44 heures (7x plus rapide!)")
print("  ✅ Batch size 4 vs 1 (4x plus efficace)")
print("  ✅ Learning rate optimisé (2e-4 vs 2e-5)")
print("  ✅ Modèle plus petit (2B vs 8B)")
print("  ✅ Mémoire réduite (2-4GB vs 25GB+)")
print("="*60)